Import the functions needed from MINGLE.

In [ ]:
from MINGLE.pp.preprocessing import read_file
from MINGLE.pl._utils import save_figure
import MINGLE as mg
import scanpy as sc
import os

Load in dataset and set path to save figure to. 

In [ ]:
OUTPUT_DIR = r"C:\Users\annet\Documents\Projects\MINGLE_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ensure save_figure writes here (scanpy-style)
sc.settings.figdir = OUTPUT_DIR

DATA_PATH = r"Z:\MINGLE\Data\Intestine\05_25_HuBMAP_tunit.csv"
adata_local = read_file(DATA_PATH)

Calculate centroids.

In [ ]:
centroids = mg.tl.centroid_Calculation(
            adata_local,
            cluster_col="Cell Type",
            neighborhood_col="Neighborhood",
        )

Calculate GMM with GPU.

In [ ]:
adata = mg.tl.gpu_gmm_probability(
            adata_local,
            centroids,
            cluster_col="Cell Type",
            neighborhood_col="Neighborhood",
            k=10,
            batch_size=20000
        )

Run positive probabilities plotting script.

In [ ]:
figs = mg.pl.edges_positive_probability(
            adata,
            prob_key="neighborhood_probabilities",
            threshold=0.25,
            save="png",
            show=True
        )

Save figure.

In [ ]:
# save figures 
if isinstance(figs, (list, tuple)) and len(figs) == 3:
    names = ["scatter", "counts_legend", "color_legend"]
else:
    names = [f"fig{i+1}" for i in range(len(figs))]

for fig, name in zip(figs, names):
    base = f"border_enrichment_{N1.replace(' ','_')}_{N2.replace(' ','_')}_{name}"
    save_figure(fig, base=base, save=True)
    print("Saved:", os.path.join(OUTPUT_DIR, base))